In [1]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
SEED = 42
EPOCHS = 30
BATCH_SIZE = 128
RESULTS_DIR = "results"

tf.keras.utils.set_random_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


TensorFlow: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
def load_data():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
    x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

    x_val, y_val = x_train[:5000], y_train[:5000]
    x_tr, y_tr = x_train[5000:], y_train[5000:]
    return (x_tr, y_tr), (x_val, y_val), (x_test, y_test)


In [3]:
def save_samples(x, y):
    plt.figure(figsize=(10, 4))
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x[i].squeeze(), cmap="gray")
        plt.title(CLASS_NAMES[y[i]], fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/samples.png", dpi=150)
    plt.close()


In [4]:
def conv_block(x, filters, dropout):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    return tf.keras.layers.Dropout(dropout)(x)


In [5]:
def build_model():
    inputs = tf.keras.layers.Input(shape=(28, 28, 1))

    x = tf.keras.layers.RandomFlip("horizontal")(inputs)
    x = tf.keras.layers.RandomTranslation(0.1, 0.1)(x)

    x = conv_block(x, 32, 0.25)
    x = conv_block(x, 64, 0.25)
    x = conv_block(x, 128, 0.25)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model



In [6]:
def save_history(history):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for a, key, title in zip(ax, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        a.plot(history.history[key], label="train")
        a.plot(history.history[f"val_{key}"], label="val")
        a.set_title(title)
        a.set_xlabel("Epoch")
        a.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150)
    plt.close()

In [7]:
def save_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Fashion-MNIST")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150)
    plt.close()


In [8]:
def save_random_predictions(model, x_test, y_test):
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(x_test), 16, replace=False)
    preds = np.argmax(model.predict(x_test[idx], verbose=0), axis=1)

    plt.figure(figsize=(10, 6))
    for i, k in enumerate(idx):
        plt.subplot(4, 4, i + 1)
        plt.imshow(x_test[k].squeeze(), cmap="gray")
        color = "green" if preds[i] == y_test[k] else "red"
        plt.title(f"P:{CLASS_NAMES[preds[i]]}\nT:{CLASS_NAMES[y_test[k]]}",
                  color=color, fontsize=7)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/sample_predictions.png", dpi=150)
    plt.close()


In [9]:
def predict(model, img, top_k=3):
    x = np.array(img, dtype="float32").reshape(1, 28, 28, 1)
    p = model.predict(x, verbose=0)[0]
    for i in np.argsort(p)[::-1][:top_k]:
        print(f"  {CLASS_NAMES[i]:12s}  {p[i] * 100:5.2f}%")



In [10]:
def main():
    (x_tr, y_tr), (x_val, y_val), (x_test, y_test) = load_data()
    print("Train:", x_tr.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)
    save_samples(x_tr, y_tr)

    model = build_model()
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=2, min_lr=1e-5),
    ]

    history = model.fit(
        x_tr, y_tr,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {test_acc:.4f}")

    y_pred = np.argmax(model.predict(x_test, batch_size=256, verbose=0), axis=1)
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)
    print(report)
    with open(f"{RESULTS_DIR}/classification_report.txt", "w") as f:
        f.write(f"Test accuracy: {test_acc:.4f}\n\n{report}")

    save_history(history)
    save_confusion_matrix(y_test, y_pred)
    save_random_predictions(model, x_test, y_test)

    model.save("fashion_cnn.keras")
    loaded = tf.keras.models.load_model("fashion_cnn.keras")
    print("\nDemo prediction (true label:", CLASS_NAMES[y_test[0]], ")")
    predict(loaded, x_test[0])


if __name__ == "__main__":
    main()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Train: (55000, 28, 28, 1) | Val: (5000, 28, 28, 1) | Test: (10000, 28, 28, 1)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 28, 28, 1)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 242,218 (946.16 KB)

 Trainable params: 241,770 (944.41 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.6512 - loss: 0.9515 - val_accuracy: 0.6400 - val_loss: 0.9275 - learning_rate: 0.0010
Epoch 2/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.7596 - loss: 0.6379 - val_accuracy: 0.8342 - val_loss: 0.4174 - learning_rate: 0.0010
Epoch 3/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.7889 - loss: 0.5652 - val_accuracy: 0.8616 - val_loss: 0.3696 - learning_rate: 0.0010
Epoch 4/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.8104 - loss: 0.5166 - val_accuracy: 0.8708 - val_loss: 0.3449 - learning_rate: 0.0010
Epoch 5/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8188 - loss: 0.4887 - val_accuracy: 0.8758 - val_loss: 0.3233 - learning_rate: 0.0010
Epoch 6/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.8299 - loss: 0.4629 - val_accuracy: 0.8826 - val_loss: 0.3165 - learning_rate: 0.0010
Epoch 7/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8395 - loss: 0